# Finetuning実験 Part 1: データ準備

**目的**: POIデータから学習用の質問-回答ペアを自動生成する

**出力**:
- `data/finetuning_train.json` - 学習データセット
- `data/finetuning_valid.json` - 検証データセット

**作成日**: 2026-01-28

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q pandas numpy tqdm
print("パッケージインストール完了")

In [ ]:
# 1.2 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
os.makedirs(DATA_DIR, exist_ok=True)
sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.3 基本ライブラリ
import json
import random
import hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
import pandas as pd
import numpy as np

random.seed(42)
np.random.seed(42)
print("ライブラリロード完了")

## Section 2: POIデータ読み込みと前処理

In [ ]:
# 2.1 POIデータ読み込み
with open(f"{DATA_DIR}/poi_documents.json", "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

# メタデータを抽出
all_pois = []
for doc in poi_documents:
    poi = doc["metadata"].copy() if "metadata" in doc else doc.copy()
    poi["content"] = doc.get("content", "")
    all_pois.append(poi)

print(f"POIデータ: {len(all_pois)}件")

In [ ]:
# 2.2 カテゴリ統計
category_counts = Counter(p.get('category', '不明') for p in all_pois)
print("カテゴリ別件数:")
for cat, count in category_counts.most_common():
    print(f"  {cat}: {count}件")

In [ ]:
# 2.3 空間情報の追加（geo_utilsから移植）
import math

SHIBUYA_STATION = {"lat": 35.658034, "lon": 139.701636}
EARTH_RADIUS_M = 6371000

def haversine_distance(lat1, lon1, lat2, lon2):
    """2点間の距離を計算（メートル）"""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return EARTH_RADIUS_M * 2 * math.asin(math.sqrt(a))

def get_direction(lat, lon, ref_lat, ref_lon):
    """方向を判定（8方位）"""
    dlat, dlon = lat - ref_lat, lon - ref_lon
    angle = math.degrees(math.atan2(dlon, dlat))
    if angle < 0: angle += 360
    directions = ["北", "北東", "東", "南東", "南", "南西", "西", "北西"]
    return directions[int((angle + 22.5) // 45) % 8]

def enrich_poi(poi):
    """POIに空間情報を追加"""
    lat = poi.get('lat')
    lon = poi.get('lon')
    if lat and lon:
        poi['distance_from_station'] = haversine_distance(
            lat, lon, SHIBUYA_STATION['lat'], SHIBUYA_STATION['lon'])
        poi['direction_from_station'] = get_direction(
            lat, lon, SHIBUYA_STATION['lat'], SHIBUYA_STATION['lon'])
        # 東西判定
        poi['is_east'] = lon > SHIBUYA_STATION['lon']
    return poi

# 全POIに空間情報を追加
enriched_pois = [enrich_poi(p.copy()) for p in all_pois]
print(f"空間情報追加完了: {len(enriched_pois)}件")

# サンプル確認
sample = enriched_pois[0]
print(f"\nサンプル: {sample.get('name')}")
print(f"  距離: {sample.get('distance_from_station', 0):.0f}m")
print(f"  方向: {sample.get('direction_from_station')}")

## Section 3: テストケースの確認（リーク防止）

In [ ]:
# 3.1 テストケースの読み込み
try:
    from src.test_cases_v2 import TEST_CASES_V2
    print(f"テストケース: {len(TEST_CASES_V2)}件")
except ImportError:
    print("警告: test_cases_v2.pyが見つかりません")
    TEST_CASES_V2 = []

In [ ]:
# 3.2 テストケースで言及されるPOI/キーワードを抽出
test_keywords = set()
test_prompts = set()

for tc in TEST_CASES_V2:
    test_prompts.add(tc.prompt.lower())
    for kw in tc.expected_keywords:
        test_keywords.add(kw.lower())

print(f"テストキーワード数: {len(test_keywords)}")
print(f"テストプロンプト数: {len(test_prompts)}")

# 特定のPOI名（テストケースで直接言及されるもの）
test_poi_names = {
    "東宝シネマ", "渋谷東武ホテル", "渋谷神南郵便局", 
    "三菱UFJ銀行渋谷支店"
}
print(f"除外POI名: {test_poi_names}")

In [ ]:
# 3.3 学習用と検証用にPOIを分割
def is_test_poi(poi):
    """テストケースで直接言及されるPOIかどうか"""
    name = poi.get('name', '')
    for test_name in test_poi_names:
        if test_name in name or name in test_name:
            return True
    return False

# 分割
train_pois = [p for p in enriched_pois if not is_test_poi(p)]
test_ref_pois = [p for p in enriched_pois if is_test_poi(p)]

print(f"学習用POI: {len(train_pois)}件")
print(f"テスト参照POI: {len(test_ref_pois)}件")

# さらに学習用を学習/検証に分割（90:10）
random.shuffle(train_pois)
split_idx = int(len(train_pois) * 0.9)
train_pois_final = train_pois[:split_idx]
valid_pois = train_pois[split_idx:]

print(f"最終学習用POI: {len(train_pois_final)}件")
print(f"検証用POI: {len(valid_pois)}件")

## Section 4: 学習データ生成

In [ ]:
# 4.1 学習データ生成クラス
@dataclass
class TrainingExample:
    """Alpaca形式の学習データ"""
    instruction: str
    input: str = ""
    output: str = ""
    pattern: str = ""  # 生成パターン（デバッグ用）

class DataGenerator:
    """学習データ生成器"""
    
    def __init__(self, pois: List[Dict]):
        self.pois = pois
        self.by_category = defaultdict(list)
        for p in pois:
            cat = p.get('category', '不明')
            self.by_category[cat].append(p)
        
        # カテゴリの日本語名マッピング
        self.category_names = {
            "飲食店/カフェ": "カフェ",
            "飲食店/レストラン": "レストラン",
            "飲食店/ファストフード": "ファストフード店",
            "飲食店/バー": "バー",
            "商店/コンビニ": "コンビニ",
            "商店/スーパー": "スーパー",
            "医療/薬局": "薬局",
            "医療/病院": "病院",
            "医療/クリニック": "クリニック",
            "金融/銀行": "銀行",
            "金融/ATM": "ATM",
            "宿泊/ホテル": "ホテル",
            "娯楽/映画館": "映画館",
            "公共/郵便局": "郵便局",
            "公共/交番": "交番",
            "交通/鉄道駅": "駅",
        }
    
    def get_category_name(self, category: str) -> str:
        """カテゴリの表示名を取得"""
        return self.category_names.get(category, category.split('/')[-1])
    
    def generate_pattern1_location(self, n: int = 200) -> List[TrainingExample]:
        """パターン1: 位置情報検索（L1相当）"""
        examples = []
        templates = [
            "{name}はどこにありますか？",
            "{name}の場所を教えてください",
            "{name}の位置情報は？",
            "{name}の座標を教えて",
        ]
        
        sampled = random.sample(self.pois, min(n, len(self.pois)))
        for poi in sampled:
            name = poi.get('name', '')
            if not name or len(name) < 2:
                continue
            
            template = random.choice(templates)
            instruction = template.format(name=name)
            
            lat = poi.get('lat', '')
            lon = poi.get('lon', '')
            distance = poi.get('distance_from_station', 0)
            direction = poi.get('direction_from_station', '')
            category = self.get_category_name(poi.get('category', ''))
            
            output = f"{name}は渋谷駅から{direction}方向に約{distance:.0f}mの場所にあります。"
            if lat and lon:
                output += f"座標は緯度{lat}、経度{lon}です。"
            if category:
                output += f"カテゴリは{category}です。"
            
            examples.append(TrainingExample(
                instruction=instruction,
                output=output,
                pattern="pattern1_location"
            ))
        
        return examples[:n]
    
    def generate_pattern2_category(self, n: int = 200) -> List[TrainingExample]:
        """パターン2: カテゴリ検索（L1相当）"""
        examples = []
        templates = [
            "渋谷駅周辺の{category}を教えてください",
            "渋谷にある{category}は？",
            "渋谷駅近くの{category}を知りたい",
            "渋谷周辺で{category}を探しています",
        ]
        
        for cat, pois_in_cat in self.by_category.items():
            if len(pois_in_cat) < 3:
                continue
            
            cat_name = self.get_category_name(cat)
            for _ in range(min(n // len(self.by_category), 20)):
                template = random.choice(templates)
                instruction = template.format(category=cat_name)
                
                # ランダムに3-5件選択
                sample_size = min(random.randint(3, 5), len(pois_in_cat))
                sampled = random.sample(pois_in_cat, sample_size)
                
                poi_list = []
                for p in sampled:
                    name = p.get('name', '不明')
                    dist = p.get('distance_from_station', 0)
                    poi_list.append(f"{name}（駅から約{dist:.0f}m）")
                
                output = f"渋谷駅周辺の{cat_name}として、{', '.join(poi_list)}があります。"
                
                examples.append(TrainingExample(
                    instruction=instruction,
                    output=output,
                    pattern="pattern2_category"
                ))
        
        return examples[:n]
    
    def generate_pattern3_nearest(self, n: int = 150) -> List[TrainingExample]:
        """パターン3: 最近傍検索（L2相当）"""
        examples = []
        templates = [
            "渋谷駅に最も近い{category}はどこですか？",
            "渋谷駅から一番近い{category}を教えて",
            "渋谷駅周辺で最寄りの{category}は？",
        ]
        
        for cat, pois_in_cat in self.by_category.items():
            if len(pois_in_cat) < 2:
                continue
            
            cat_name = self.get_category_name(cat)
            
            # 距離でソート
            sorted_pois = sorted(
                [p for p in pois_in_cat if p.get('distance_from_station')],
                key=lambda x: x.get('distance_from_station', float('inf'))
            )
            
            if not sorted_pois:
                continue
            
            nearest = sorted_pois[0]
            
            for _ in range(min(n // len(self.by_category), 10)):
                template = random.choice(templates)
                instruction = template.format(category=cat_name)
                
                name = nearest.get('name', '不明')
                dist = nearest.get('distance_from_station', 0)
                direction = nearest.get('direction_from_station', '')
                lat = nearest.get('lat', '')
                lon = nearest.get('lon', '')
                
                output = f"渋谷駅に最も近い{cat_name}は{name}で、駅から{direction}方向に約{dist:.0f}mの距離にあります。"
                if lat and lon:
                    output += f"座標は({lat}, {lon})です。"
                
                # 2番目に近いものも言及
                if len(sorted_pois) > 1:
                    second = sorted_pois[1]
                    output += f"次に近いのは{second.get('name', '不明')}（約{second.get('distance_from_station', 0):.0f}m）です。"
                
                examples.append(TrainingExample(
                    instruction=instruction,
                    output=output,
                    pattern="pattern3_nearest"
                ))
        
        return examples[:n]
    
    def generate_pattern4_comparison(self, n: int = 150) -> List[TrainingExample]:
        """パターン4: 東西比較（L2相当）"""
        examples = []
        templates = [
            "渋谷駅の東側と西側、どちらに{category}が多いですか？",
            "{category}は渋谷駅の東と西、どちらに多い？",
            "渋谷駅周辺の{category}は東側と西側どちらが充実していますか？",
        ]
        
        for cat, pois_in_cat in self.by_category.items():
            if len(pois_in_cat) < 5:
                continue
            
            cat_name = self.get_category_name(cat)
            
            east_pois = [p for p in pois_in_cat if p.get('is_east', False)]
            west_pois = [p for p in pois_in_cat if not p.get('is_east', True)]
            
            east_count = len(east_pois)
            west_count = len(west_pois)
            
            for _ in range(min(n // len(self.by_category), 10)):
                template = random.choice(templates)
                instruction = template.format(category=cat_name)
                
                if east_count > west_count:
                    output = f"渋谷駅の東側に{cat_name}が多く、{east_count}件あります。西側は{west_count}件です。"
                elif west_count > east_count:
                    output = f"渋谷駅の西側に{cat_name}が多く、{west_count}件あります。東側は{east_count}件です。"
                else:
                    output = f"渋谷駅の東側と西側で{cat_name}の数はほぼ同じで、それぞれ{east_count}件です。"
                
                examples.append(TrainingExample(
                    instruction=instruction,
                    output=output,
                    pattern="pattern4_comparison"
                ))
        
        return examples[:n]
    
    def generate_pattern5_aggregation(self, n: int = 150) -> List[TrainingExample]:
        """パターン5: 集約（L2相当）"""
        examples = []
        templates = [
            "渋谷駅から{radius}m以内に{category}はいくつありますか？",
            "渋谷駅周辺{radius}m以内の{category}の数は？",
            "渋谷駅から{radius}m圏内にある{category}を数えてください",
        ]
        radii = [200, 300, 500, 800]
        
        for cat, pois_in_cat in self.by_category.items():
            if len(pois_in_cat) < 3:
                continue
            
            cat_name = self.get_category_name(cat)
            
            for radius in radii:
                pois_in_radius = [
                    p for p in pois_in_cat 
                    if p.get('distance_from_station', float('inf')) <= radius
                ]
                count = len(pois_in_radius)
                
                if count == 0:
                    continue
                
                template = random.choice(templates)
                instruction = template.format(radius=radius, category=cat_name)
                
                output = f"渋谷駅から{radius}m以内には{count}件の{cat_name}があります。"
                if count <= 5:
                    names = [p.get('name', '不明') for p in pois_in_radius]
                    output += f"具体的には、{', '.join(names)}です。"
                
                examples.append(TrainingExample(
                    instruction=instruction,
                    output=output,
                    pattern="pattern5_aggregation"
                ))
        
        return examples[:n]
    
    def generate_pattern6_constraint(self, n: int = 100) -> List[TrainingExample]:
        """パターン6: 制約付き検索（L3相当）"""
        examples = []
        
        # 距離制約
        for cat, pois_in_cat in self.by_category.items():
            if len(pois_in_cat) < 3:
                continue
            
            cat_name = self.get_category_name(cat)
            
            for radius in [300, 500]:
                pois_in_radius = [
                    p for p in pois_in_cat 
                    if p.get('distance_from_station', float('inf')) <= radius
                ]
                
                if not pois_in_radius:
                    continue
                
                instruction = f"渋谷駅から{radius}m以内にある{cat_name}を教えてください"
                
                poi_info = []
                for p in pois_in_radius[:5]:
                    name = p.get('name', '不明')
                    dist = p.get('distance_from_station', 0)
                    poi_info.append(f"{name}（約{dist:.0f}m）")
                
                output = f"渋谷駅から{radius}m以内の{cat_name}として、{', '.join(poi_info)}があります。"
                if len(pois_in_radius) > 5:
                    output += f"他にも{len(pois_in_radius) - 5}件あります。"
                
                examples.append(TrainingExample(
                    instruction=instruction,
                    output=output,
                    pattern="pattern6_constraint"
                ))
        
        return examples[:n]
    
    def generate_pattern7_reasoning(self, n: int = 50) -> List[TrainingExample]:
        """パターン7: 推論・判断（L4相当）"""
        examples = []
        
        # カテゴリ密度に基づく出店判断
        for cat, pois_in_cat in self.by_category.items():
            cat_name = self.get_category_name(cat)
            count = len(pois_in_cat)
            
            if count < 3:
                continue
            
            # 500m以内の件数
            count_500m = len([
                p for p in pois_in_cat 
                if p.get('distance_from_station', float('inf')) <= 500
            ])
            
            instruction = f"渋谷駅周辺で新規{cat_name}の出店を検討しています。競合状況を教えてください。"
            
            if count_500m > 10:
                output = f"渋谷駅から500m以内に{count_500m}件の{cat_name}があり、競合が非常に多いエリアです。差別化戦略が必要でしょう。"
            elif count_500m > 5:
                output = f"渋谷駅から500m以内に{count_500m}件の{cat_name}があり、競合は中程度です。立地選定が重要になります。"
            else:
                output = f"渋谷駅から500m以内に{count_500m}件の{cat_name}しかなく、出店余地があると考えられます。"
            
            # 東西の分布も追加
            east = len([p for p in pois_in_cat if p.get('is_east', False)])
            west = count - east
            if east > west * 1.5:
                output += f"西側（{west}件）に比べ東側（{east}件）に集中しているため、西側への出店も検討に値します。"
            elif west > east * 1.5:
                output += f"東側（{east}件）に比べ西側（{west}件）に集中しているため、東側への出店も検討に値します。"
            
            examples.append(TrainingExample(
                instruction=instruction,
                output=output,
                pattern="pattern7_reasoning"
            ))
        
        return examples[:n]
    
    def generate_pattern8_sensitivity(self, n: int = 50) -> List[TrainingExample]:
        """パターン8: 感度分析（L5相当）"""
        examples = []
        
        for cat, pois_in_cat in self.by_category.items():
            if len(pois_in_cat) < 5:
                continue
            
            cat_name = self.get_category_name(cat)
            
            # 異なる半径での件数
            count_300 = len([p for p in pois_in_cat if p.get('distance_from_station', float('inf')) <= 300])
            count_500 = len([p for p in pois_in_cat if p.get('distance_from_station', float('inf')) <= 500])
            count_800 = len([p for p in pois_in_cat if p.get('distance_from_station', float('inf')) <= 800])
            
            instruction = f"検索範囲を300mから500mに広げると、{cat_name}の検索結果はどう変わりますか？"
            
            diff = count_500 - count_300
            ratio = count_500 / count_300 if count_300 > 0 else float('inf')
            
            output = f"300m以内では{count_300}件、500m以内では{count_500}件となり、{diff}件増加します。"
            
            if ratio >= 2:
                output += f"件数が{ratio:.1f}倍に増えるため、検索範囲によって結果が大きく変わります。"
            elif ratio >= 1.5:
                output += f"件数が{ratio:.1f}倍に増えますが、傾向は維持されます。"
            else:
                output += f"増加率は{ratio:.1f}倍で、範囲を広げても大きな変化はありません。"
            
            examples.append(TrainingExample(
                instruction=instruction,
                output=output,
                pattern="pattern8_sensitivity"
            ))
        
        return examples[:n]
    
    def generate_all(self) -> List[TrainingExample]:
        """すべてのパターンでデータを生成"""
        all_examples = []
        
        all_examples.extend(self.generate_pattern1_location(200))
        all_examples.extend(self.generate_pattern2_category(200))
        all_examples.extend(self.generate_pattern3_nearest(150))
        all_examples.extend(self.generate_pattern4_comparison(150))
        all_examples.extend(self.generate_pattern5_aggregation(150))
        all_examples.extend(self.generate_pattern6_constraint(100))
        all_examples.extend(self.generate_pattern7_reasoning(50))
        all_examples.extend(self.generate_pattern8_sensitivity(50))
        
        return all_examples

print("DataGeneratorクラス定義完了")

In [ ]:
# 4.2 学習データ生成
print("=== 学習データ生成 ===")

generator = DataGenerator(train_pois_final)
train_examples = generator.generate_all()

print(f"\n生成データ数: {len(train_examples)}件")

# パターン別統計
pattern_counts = Counter(ex.pattern for ex in train_examples)
print("\nパターン別:")
for pattern, count in pattern_counts.most_common():
    print(f"  {pattern}: {count}件")

In [ ]:
# 4.3 検証データ生成
print("=== 検証データ生成 ===")

valid_generator = DataGenerator(valid_pois)
valid_examples = valid_generator.generate_all()

# 検証データは各パターン20件まで
valid_by_pattern = defaultdict(list)
for ex in valid_examples:
    valid_by_pattern[ex.pattern].append(ex)

valid_examples_limited = []
for pattern, examples in valid_by_pattern.items():
    valid_examples_limited.extend(examples[:20])

print(f"検証データ数: {len(valid_examples_limited)}件")

## Section 5: データ品質検証

In [ ]:
# 5.1 サンプル確認
print("=== サンプルデータ確認 ===")

for pattern in sorted(set(ex.pattern for ex in train_examples)):
    examples = [ex for ex in train_examples if ex.pattern == pattern]
    sample = random.choice(examples)
    print(f"\n[{pattern}]")
    print(f"Q: {sample.instruction}")
    print(f"A: {sample.output[:200]}..." if len(sample.output) > 200 else f"A: {sample.output}")

In [ ]:
# 5.2 テストリークチェック
print("=== テストリークチェック ===")

def check_similarity(text1: str, text2: str) -> float:
    """簡易的な類似度チェック（Jaccard係数）"""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    if not words1 or not words2:
        return 0.0
    return len(words1 & words2) / len(words1 | words2)

leak_candidates = []
for ex in train_examples:
    for tc in TEST_CASES_V2:
        sim = check_similarity(ex.instruction, tc.prompt)
        if sim > 0.5:
            leak_candidates.append((ex.instruction, tc.prompt, sim))

if leak_candidates:
    print(f"リーク候補: {len(leak_candidates)}件")
    for train_q, test_q, sim in leak_candidates[:5]:
        print(f"  類似度 {sim:.2f}:")
        print(f"    学習: {train_q[:50]}...")
        print(f"    テスト: {test_q[:50]}...")
else:
    print("リーク候補なし")

In [ ]:
# 5.3 重複チェック
print("=== 重複チェック ===")

instruction_hashes = set()
duplicates = []

for ex in train_examples:
    h = hashlib.md5(ex.instruction.encode()).hexdigest()
    if h in instruction_hashes:
        duplicates.append(ex.instruction)
    instruction_hashes.add(h)

print(f"重複: {len(duplicates)}件")
if duplicates:
    print("重複サンプル:")
    for d in duplicates[:3]:
        print(f"  {d[:50]}...")

In [ ]:
# 5.4 重複を除去
seen_instructions = set()
unique_train_examples = []

for ex in train_examples:
    if ex.instruction not in seen_instructions:
        seen_instructions.add(ex.instruction)
        unique_train_examples.append(ex)

print(f"重複除去後: {len(unique_train_examples)}件（元: {len(train_examples)}件）")
train_examples = unique_train_examples

## Section 6: データ保存

In [ ]:
# 6.1 Alpaca形式でJSON保存
def examples_to_alpaca(examples: List[TrainingExample]) -> List[Dict]:
    """TrainingExampleをAlpaca形式に変換"""
    return [
        {
            "instruction": ex.instruction,
            "input": ex.input,
            "output": ex.output
        }
        for ex in examples
    ]

# 学習データ保存
train_data = examples_to_alpaca(train_examples)
train_path = f"{DATA_DIR}/finetuning_train.json"
with open(train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
print(f"学習データ保存: {train_path} ({len(train_data)}件)")

# 検証データ保存
valid_data = examples_to_alpaca(valid_examples_limited)
valid_path = f"{DATA_DIR}/finetuning_valid.json"
with open(valid_path, "w", encoding="utf-8") as f:
    json.dump(valid_data, f, ensure_ascii=False, indent=2)
print(f"検証データ保存: {valid_path} ({len(valid_data)}件)")

In [ ]:
# 6.2 メタデータ保存
metadata = {
    "created_at": pd.Timestamp.now().isoformat(),
    "total_pois": len(all_pois),
    "train_pois": len(train_pois_final),
    "valid_pois": len(valid_pois),
    "test_ref_pois": len(test_ref_pois),
    "train_examples": len(train_data),
    "valid_examples": len(valid_data),
    "pattern_distribution": dict(pattern_counts),
    "leak_candidates": len(leak_candidates),
    "duplicates_removed": len(duplicates)
}

meta_path = f"{DATA_DIR}/finetuning_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"メタデータ保存: {meta_path}")

print("\n=== メタデータ ===")
for k, v in metadata.items():
    print(f"  {k}: {v}")

## Section 7: 最終確認

In [ ]:
# 7.1 保存ファイル確認
print("=== 保存ファイル確認 ===")
import os

files = [
    f"{DATA_DIR}/finetuning_train.json",
    f"{DATA_DIR}/finetuning_valid.json",
    f"{DATA_DIR}/finetuning_metadata.json"
]

for f in files:
    if os.path.exists(f):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {os.path.basename(f)}: {size_kb:.1f} KB")
    else:
        print(f"  {os.path.basename(f)}: 見つかりません")

In [ ]:
# 7.2 サマリー
print("="*50)
print("データ準備完了")
print("="*50)
print(f"\n学習データ: {len(train_data)}件")
print(f"検証データ: {len(valid_data)}件")
print(f"\n次のステップ: finetuning_02_training.ipynb でQLoRAファインチューニングを実行")